# Direct Phoneme -> Text Baseline (no LLM, no context)

A minimal **GRU encoder-decoder + Bahdanau attention** trained from scratch on the LRS2 phoneme->text corpus. Purpose: establish the *floor* for the thesis comparison.

Trains in ~15-20 min on T4 or ~60 min on Colab CPU. Designed for the free Colab tier.

**v2 note:** the original GRU decoder without attention collapsed into 5-char cycles. This version adds Bahdanau cross-attention so the decoder can look at every encoder time-step.

## 1. Setup -- mount Drive and confirm data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/P2T/data'
CORPUS_CSV = f'{DATA_DIR}/sentphonemepairs_LRS2_original.csv'

import os
assert os.path.isfile(CORPUS_CSV), (
    f'Could not find {CORPUS_CSV}. ',
    f'Files in {DATA_DIR}: {os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else chr(60)+"dir missing"+chr(62)}',
)
print(f'Using corpus: {CORPUS_CSV}')
print(f'Corpus size: {os.path.getsize(CORPUS_CSV) / 1e6:.2f} MB')

OUT_DIR = '/content/drive/MyDrive/P2T/direct_baseline_out'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR}')

## 2. Install/check dependencies (preinstalled on Colab)

torch / numpy are preinstalled. We use only standard library + these -- no transformers, no peft, no bitsandbytes.

In [ ]:
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 3. Hyperparameters

Small enough to train end-to-end on Colab free tier within an hour.

In [ ]:
# Default (fits Colab free tier, ~15-20 min on T4)
N_PAIRS        = 5000     # 0 = full 48k
MAX_TRAIN      = 4000     # absolute cap on train rows after 80/20 split
EMB_DIM        = 64
HID_DIM        = 128
N_LAYERS       = 1
DROPOUT        = 0.2
BATCH_SIZE     = 32
N_EPOCHS       = 8
LR             = 3e-3
TEACHER_FORCE_P = 0.5
MAX_TEXT_LEN   = 80
PHONEME_VOCAB_MAX = 200
TEXT_VOCAB_MAX    = 200

# Scale-up (T4, ~1-2 h): uncomment
# N_PAIRS, MAX_TRAIN = 0, 30000
# EMB_DIM, HID_DIM, BATCH_SIZE = 128, 256, 64
# N_EPOCHS = 12

## 4. The model (GRU + Bahdanau attention)

Bidirectional GRU encoder reads ARPAbet phoneme tokens; attention-equipped GRU decoder writes English characters. Vocabularies are built from the training set on first call.

In [ ]:
import csv, random, re, time
from collections import Counter
from typing import List, Tuple
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PAD, BOS, EOS, UNK = 0, 1, 2, 3

def clean_phonemes(raw):
    raw = (raw or '').strip()
    raw = re.sub(r'<SOS>|<EOS>', '', raw)
    raw = re.sub(r'[012]', '', raw)
    raw = raw.replace('<space>', ' ')
    return re.sub(r'\s+', ' ', raw).strip()

def clean_text(raw):
    raw = (raw or '').lower()
    raw = re.sub(r"[^a-z ']+", ' ', raw)
    return re.sub(r'\s+', ' ', raw).strip()

phoneme_vocab, text_vocab = {}, {}
phoneme_inv, text_inv = {}, {}

def build_vocabs(train_pairs):
    global phoneme_vocab, text_vocab, phoneme_inv, text_inv
    phonemes = Counter(p for p, _ in train_pairs for p in p.split())
    texts    = Counter(c for _, t in train_pairs for c in t)
    phoneme_vocab = {'<PAD>': PAD, '<BOS>': BOS, '<EOS>': EOS, '<UNK>': UNK}
    for p, _ in phonemes.most_common(PHONEME_VOCAB_MAX - 4):
        phoneme_vocab.setdefault(p, len(phoneme_vocab))
    text_vocab = {'<PAD>': PAD, '<BOS>': BOS, '<EOS>': EOS, '<UNK>': UNK, ' ': len(text_vocab)}
    for c, _ in texts.most_common(TEXT_VOCAB_MAX - 5):
        if c == ' ':
            continue
        text_vocab.setdefault(c, len(text_vocab))
    phoneme_inv = {v: k for k, v in phoneme_vocab.items()}
    text_inv    = {v: k for k, v in text_vocab.items()}
    print(f'  Vocab: phonemes={len(phoneme_vocab)}  text_chars={len(text_vocab)}')

def encode_phonemes(s):
    return [phoneme_vocab.get(t, phoneme_vocab['<UNK>']) for t in s.split()]

def encode_text(s):
    return [text_vocab.get(c, text_vocab['<UNK>']) for c in s]

class PhonemeTextDataset(Dataset):
    def __init__(self, pairs):
        self.src = [encode_phonemes(p) for p, _ in pairs]
        self.tgt_in, self.tgt_out = [], []
        for _, t in pairs:
            ids = encode_text(t)[: MAX_TEXT_LEN - 2]
            self.tgt_in.append([BOS] + ids)
            self.tgt_out.append(ids + [EOS])
    def __len__(self): return len(self.src)
    def __getitem__(self, i): return self.src[i], self.tgt_in[i], self.tgt_out[i]

def collate(batch):
    srcs, tgts_in, tgts_out = zip(*batch)
    def pad(seqs, pad_id=PAD):
        m = max(len(s) for s in seqs)
        return torch.tensor([s + [pad_id]*(m-len(s)) for s in seqs], dtype=torch.long)
    src   = pad(srcs)
    tgt_i = pad(tgts_in)
    tgt_o = pad(tgts_out)
    mask  = (src == PAD)
    return src, tgt_i, tgt_o, mask

class Encoder(nn.Module):
    """Bidirectional GRU encoder. Returns per-token outputs so the
    decoder can attend over every position."""
    def __init__(self, vocab, emb, hid, n_layers, drop):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.rnn = nn.GRU(emb, hid, n_layers,
                          dropout=drop if n_layers > 1 else 0.0,
                          batch_first=True, bidirectional=True)
        self.hid = hid; self.n_layers = n_layers
    def forward(self, src, src_mask):
        emb = self.emb(src)
        out, _ = self.rnn(emb)
        return out

class BahdanauAttention(nn.Module):
    """Additive attention: e_t = v^T tanh(W_h dec_h + W_e enc_out)."""
    def __init__(self, enc_dim, dec_dim, attn_dim):
        super().__init__()
        self.W_h = nn.Linear(dec_dim, attn_dim, bias=False)
        self.W_e = nn.Linear(enc_dim, attn_dim, bias=False)
        self.v   = nn.Linear(attn_dim, 1, bias=False)
    def forward(self, dec_h, enc_outs, src_mask):
        scores = self.v(torch.tanh(self.W_h(dec_h).unsqueeze(1) + self.W_e(enc_outs))).squeeze(-1)
        scores = scores.masked_fill(src_mask, float('-inf'))
        alpha  = F.softmax(scores, dim=-1)
        ctx    = torch.bmm(alpha.unsqueeze(1), enc_outs).squeeze(1)
        return ctx, alpha

class AttnDecoder(nn.Module):
    """1-layer GRU decoder with cross-attention to encoder outputs."""
    def __init__(self, vocab, emb, dec_hid, enc_dim, n_layers, drop):
        super().__init__()
        self.emb  = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.attn = BahdanauAttention(enc_dim, dec_hid, attn_dim=dec_hid)
        self.rnn  = nn.GRU(emb + enc_dim, dec_hid, n_layers,
                           dropout=drop if n_layers > 1 else 0.0,
                           batch_first=True)
        self.fc   = nn.Linear(dec_hid + enc_dim + emb, vocab)
        self.dec_hid = dec_hid; self.n_layers = n_layers
    def forward(self, tgt_in, dec_h0, enc_outs, src_mask):
        B, Td = tgt_in.shape
        emb = self.emb(tgt_in)
        outputs = []
        h = dec_h0.unsqueeze(0)
        for t in range(Td):
            ctx, _ = self.attn(h.squeeze(0), enc_outs, src_mask)
            rnn_in = torch.cat([emb[:, t, :], ctx], dim=-1).unsqueeze(1)
            out, h = self.rnn(rnn_in, h)
            out_t  = self.fc(torch.cat([out.squeeze(1), ctx, emb[:, t, :]], dim=-1))
            outputs.append(out_t)
        return torch.stack(outputs, dim=1)

class Seq2Seq(nn.Module):
    def __init__(self, src_v, tgt_v):
        super().__init__()
        self.enc = Encoder(src_v, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
        self.bridge = nn.Linear(2 * HID_DIM, HID_DIM)
        self.dec    = AttnDecoder(tgt_v, EMB_DIM, HID_DIM,
                                  enc_dim=2 * HID_DIM,
                                  n_layers=N_LAYERS, drop=DROPOUT)
    def forward(self, src, src_mask, tgt_in):
        enc_outs = self.enc(src, src_mask)
        ctx_init = enc_outs.mean(dim=1)
        dec_h0   = self.bridge(ctx_init)
        return self.dec(tgt_in, dec_h0, enc_outs, src_mask)

@torch.no_grad()
def greedy_decode(model, src, max_len=MAX_TEXT_LEN):
    model.eval()
    src = src.unsqueeze(0).to(DEVICE)
    mask = (src == PAD)
    enc_outs = model.enc(src, mask)
    ctx_init = enc_outs.mean(dim=1)
    dec_h0   = model.bridge(ctx_init)
    y = torch.tensor([[BOS]], device=DEVICE)
    out_ids = []
    for _ in range(max_len):
        logits = model.dec(y, dec_h0, enc_outs, mask)[:, -1, :]
        nxt = int(logits.argmax(-1).item())
        if nxt == EOS:
            break
        out_ids.append(nxt)
        y = torch.cat([y, torch.tensor([[nxt]], device=DEVICE)], dim=1)
    return out_ids

def detok(ids):
    return ''.join(text_inv.get(i, '?') for i in ids)

def edit_distance(a, b):
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[n][m], n

## 5. Data loading + 80/20 split

In [ ]:
def load_pairs(n):
    rows = []
    with open(CORPUS_CSV, 'r', encoding='utf-8', newline='') as f:
        for sent, phon in csv.reader(f):
            if not sent or not phon: continue
            cs = clean_text(sent); cp = clean_phonemes(phon)
            if cs and cp: rows.append((cp, cs))
            if n and len(rows) >= n: break
    cut = int(len(rows) * 0.8)
    return rows[:cut], rows[cut:]

print('Loading + cleaning CSV...')
t0 = time.time()
train_pairs, val_pairs = load_pairs(N_PAIRS)
train_pairs = train_pairs[:MAX_TRAIN]
print(f'  Loaded train={len(train_pairs):,}  val={len(val_pairs):,}  ({time.time()-t0:.1f}s)')
build_vocabs(train_pairs)

train_ds = PhonemeTextDataset(train_pairs)
val_ds   = PhonemeTextDataset(val_pairs)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=0)
print(f'\nDataset: {len(train_ds):,} train / {len(val_ds):,} val')

## 6. Train

Scheduled-sampling-lite: 50% teacher forcing / 50% model predictions. Adam @ 3e-3, gradient clipping 1.0.

In [ ]:
def train_one_epoch(model, opt, loader):
    model.train()
    total, n = 0.0, 0
    for src, tgt_in, tgt_out, src_mask in loader:
        src      = src.to(DEVICE)
        tgt_in   = tgt_in.to(DEVICE)
        tgt_out  = tgt_out.to(DEVICE)
        src_mask = src_mask.to(DEVICE)
        if random.random() < TEACHER_FORCE_P:
            inp = tgt_in
        else:
            # Scheduled sampling: feed previous prediction back.
            with torch.no_grad():
                enc_outs = model.enc(src, src_mask)
                ctx_init = enc_outs.mean(dim=1)
                dec_h    = model.bridge(ctx_init)
                y = tgt_in[:, :1]
                preds = []
                for t in range(tgt_in.size(1) - 1):
                    logits = model.dec(y, dec_h, enc_outs, src_mask)[:, -1, :]
                    nxt    = logits.argmax(-1, keepdim=True)
                    preds.append(nxt)
                    y      = torch.cat([y, nxt], dim=1)
                inp = torch.cat([tgt_in[:, :1]] + preds, dim=1)[:, :tgt_in.size(1)]
        logits = model(src, src_mask, inp)
        loss   = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                 tgt_out.reshape(-1), ignore_index=PAD)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item() * src.size(0); n += src.size(0)
    return total / n

def evaluate(model, ds):
    model.eval()
    wer_d, wer_n = 0, 0
    cer_d, cer_n = 0, 0
    em_ok = 0
    n = len(ds)
    sample_idx = list(range(n)) if n <= 2000 else random.sample(range(n), 500)
    for i in sample_idx:
        src, _, _ = ds[i]
        ids = greedy_decode(model, torch.tensor(src))
        pred = detok(ids)
        ref  = detok(ds.tgt_out[i][:-1])
        ed, nn_ = edit_distance(ref.split(), pred.split())
        wer_d += ed; wer_n += max(1, nn_)
        ed, nn_ = edit_distance(list(ref), list(pred))
        cer_d += ed; cer_n += max(1, nn_)
        if pred.strip() == ref.strip(): em_ok += 1
    return {'n': len(sample_idx), 'WER': wer_d/wer_n if wer_n else 0.0,
            'CER': cer_d/cer_n if cer_n else 0.0, 'EM': em_ok/len(sample_idx)}

model = Seq2Seq(len(phoneme_vocab), len(text_vocab)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: GRU+attn, {n_params:,} parameters')
opt = torch.optim.Adam(model.parameters(), lr=LR)

print(f"\n{'epoch':>5} {'train_loss':>12} {'val_WER':>9} {'val_CER':>9} {'val_EM':>9}  {'time':>7}")
print('-' * 60)
for epoch in range(1, N_EPOCHS + 1):
    ep_t = time.time()
    loss = train_one_epoch(model, opt, train_loader)
    m = evaluate(model, val_ds)
    print(f"{epoch:>5} {loss:>12.4f} {m['WER']*100:>8.2f}% {m['CER']*100:>8.2f}% {m['EM']*100:>8.2f}%  {time.time()-ep_t:>6.1f}s")
print('-' * 60)

## 7. Examples + final metrics

Sample 8 val rows and write the summary CSV to Drive.

In [ ]:
print('\nFinal examples (val):')
for i in random.sample(range(len(val_ds)), min(8, len(val_ds))):
    src, _, _ = val_ds[i]
    ids = greedy_decode(model, torch.tensor(src))
    print(f"  phon: {' '.join(phoneme_inv.get(x, '?') for x in src)}")
    print(f"  ref : {detok(val_ds.tgt_out[i][:-1])!r}")
    print(f"  pred: {detok(ids)!r}")
    print()

summary_path = os.path.join(OUT_DIR, 'direct_baseline_metrics.csv')
with open(summary_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['model', 'n_val', 'WER', 'CER', 'EM', 'n_params', 'device'])
    m = evaluate(model, val_ds)
    w.writerow(['GRU-direct-attn', m['n'], f"{m['WER']*100:.4f}", f"{m['CER']*100:.4f}", f"{m['EM']*100:.4f}", n_params, str(DEVICE)])
print(f'Metrics written to {summary_path}')

## Done

- `direct_baseline_metrics.csv` is in `MyDrive/P2T/direct_baseline_out/`.
- Drop those numbers into the thesis comparison table:

| Method | WER | CER | EM |
|---|---|---|---|
| Zero-shot Llama-3.2-3B | _(from `zero_shot_baseline.py`)_ | | |
| Direct GRU+attn (this notebook) | _your number_ | _your number_ | _your number_ |
| LoRA CPT decoder | _(thesis number)_ | | |

Even with attention, expect this row to land in the 60-90% WER range -- still 5-20x worse than zero-shot Llama, which is the point: it quantifies the LLM prior itself.